# DSC 550 Term Project

## [Milestone 1](./6.2_milestone_1.ipynb) 

Copy and paste later

## Milestone 2

Now that you have created your idea, located data, and have started your graphical analysis, you will move on to the data preparation process of your project. After completing Milestone 2, your data should be ready for the model building/evaluation phase.

Here is a list of steps to consider performing in Milestone 2:

- Drop any features that are not useful for your model building and explain why they are not useful.
- Perform any data extraction/selection steps.
- Transform features if necessary.
- Engineer new useful features.
- Deal with missing data (do not just drop rows or columns without justifying this).
- Create dummy variables if necessary.

Explain your process at each step. You can use any methods/tools you think are most appropriate. Do what makes the most sense for your data/problem. This will vary greatly among different projects. Be careful to avoid data snooping in these steps.

It is important to note that these milestones are meant to keep you on track for the final project submission. At any point, you can pivot or modify your project as needed based on what you discover. These milestones are not final versions; they are drafts of the many steps you need to complete along the way.

As a reminder, Teams is a great place to discuss your project with your peers. Feel free to solicit feedback/input (without creating a group project!) and collaborate on your projects with your peers.

Each milestone will build on top of each other, so make sure you do not fall behind. Submit Milestones 1 & 2 together. I recommend building your project milestones in a Jupyter Notebook, building upon one another. However, make sure it is clear where Milestone 1 ends and Milestone 2 begins.

In [1]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score, 
    recall_score, 
    f1_score, 
    roc_curve, 
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.naive_bayes import MultinomialNB

**1. Drop any features that are not useful for your model building and explain why they are not useful.**

In [2]:
csv = ["excluded", "included", "irrelevant"]

for x in csv:
    df = pd.read_csv(f"./data/{x}.csv", encoding = "utf-8") # Import the exported PubMed/MEDLINE results from search.
    df = df.drop(columns = ["Published Month", "Issue", "Pages", "Accession Number", "Ref", "Covidence #", "Notes", "Tags", "Volume", "Authors"])
    globals()[x] = df

The following features from PubMed/MEDLINE export were dropped: Published Month, Issue, Pages, Accession Number, Ref, Covidence #, Notes, Tags, and Volume. These features were not useful for the building the predictive model because since the goal is to use text classification for the Title and Abstract columns in order to find the words and phrases that best predict whether the study should be included or excluded from the review according to a predefined eligbility criteria for inclusion and exclusion of studies. The first step of the screening process involves Title and Abstract screening, and thus, only these two features were selected for building the model.

**2. Perform any data extraction/selection steps**: select features Title, Abstract, Published Year, and Journal, and code results Included, Excluded as 0, 1 as a Label column.

In [10]:
irrelevant["Label"] = 0 # Excluded during title and abstract screening
excluded["Label"] = 1 # Excluded during full-text screening
included["Label"] = 2 # Included into qualitative review

df = pd.DataFrame(columns = ["Study", "Title", "Abstract", "Published Year", "DOI", "Journal", "Label"])
df = pd.concat([df, included, excluded, irrelevant])
df.to_csv("./data/labeled_training_data.csv", encoding = "utf-8")

**3. Transform features if necessary**: create stemmed Title and Abstract columns using NLTK's PorterStemmer

In [11]:
from nltk.stem import PorterStemmer

PS = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS)

def stemmer(column):
    words = re.sub(r"[^a-zA-Z]", " ", str(column).lower()).split()
    return " ".join(PS.stem(word) for word in words if word not in stop_words)

df["stemmed_title"] = df["Title"].apply(stemmer)
df["stemmed_abstract"] = df["Abstract"].apply(stemmer)

In [12]:
df.to_csv("./data/stemmed_training.csv", encoding = "utf-8")

**4. Deal with missing data (do not just drop rows or columns without justifying this).**

In [6]:
df = pd.read_csv("./data/stemmed_training.csv", encoding = "utf-8")
df["stemmed_title"] = df["stemmed_title"].fillna("")
df["stemmed_abstract"] = df["stemmed_abstract"].fillna("")
df["DOI"] = df["DOI"].fillna("")

**5. Engineer new useful features:** the stemmed title and stemmed abstract features were combined into one feature column.

In [7]:
df["stemmed_title_abstract"] = (df["stemmed_title"] + " " + df["stemmed_abstract"])

**6. Create dummy variables if necessary.**

**Create training and test datasets**

In [8]:
df = df[df["Label"] < 2] # Include only title and abstract screening labels by dropping rows that were excluded / included during full-text screening 
df["Label"] = df["Label"].astype(int)

X = df["stemmed_title_abstract"]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, 
    random_state=42, 
    stratify=df["Label"]
)

**Create predictive model**

In [14]:
vectorizer = TfidfVectorizer(
#    stop_words = "english",
    ngram_range=(1, 2),
#    min_df = 2,
#    max_df = 0.
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

lr = LogisticRegression(max_iter = 1000)
model = lr.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test)

AttributeError: 'csr_matrix' object has no attribute 'lower'

**Most frequently used words/phrases associated with included and excluded studies**

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = pd.Series(model.coef_[0], index = feature_names)

**Included**

In [ ]:
coef.sort_values(ascending = False).head(50) # Included studies

**Excluded**

In [ ]:
coef.sort_values(ascending = True).head(50) # Excluded studies

## Milestone 3


**In Milestone 3, you will begin the process of selecting, building, and evaluating a model.**

You are required to train and evaluate at least one model in this milestone. Write step-by-step for performing each of these steps. You can use any methods/tools you think are most appropriate, but you should explain/justify why you are selecting the model(s) and evaluation metric(s) you choose. It is important to think about what type of model and metric makes sense for your problem. Again, do what makes the most sense for your project. Write a short overview/conclusion of the insights gained from your model building/evaluation.

Logistic or linear regression model was selected to 

It is important to note that these milestones are meant to keep you on track for the final project submission. At any point, you can pivot or modify your project as needed based on what you discover. These milestones are not final versions; they are drafts of the many steps you need to complete along the way.

**As a reminder, Teams is a great place to discuss your project with your peers. Feel free to solicit feedback/input (without creating a group project!) and collaborate on your projects with your peers.**


Each milestone will build on top of each other, so make sure you do not fall behind. Submit Milestones 1-3 together. I recommend building your project milestones in a Jupyter Notebook, building upon one another. However, make sure it is clear where each milestone begins and ends.

